# 29 — Deep Binary Hashing 512 Bits

A neural hashing baseline. It trains a 512-bit code with a straight-through estimator, using WJ positives and random negatives, then ranks by Hamming similarity.

This is not expected to be the best WJ method, but it helps position your continuous simplex MLP against deep hashing literature.


In [1]:
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup,
    eval_recall,
    l1_simplex,
    load_dataset,
    nmslib_neighbors,
    rerank_raw_wj_numpy,
    save_result,
)

# Edit here
dataset_name = "10k"
out_dim = 512
device_str = "cuda:0"
device = torch.device(device_str if torch.cuda.is_available() else "cpu")
THREADS = 32
seed = 42
batch_size = 512
epochs = 30
lr = 1e-3
weight_decay = 1e-4
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]
run_rerank = True

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print(f"device={device}")

METHOD_NAME = "deep_binary_hash_wj_512"
NOTEBOOK_NAME = "29_deep_binary_hash_wj_512.ipynb"
OUT_PATH = "/tmp/results_sota_deep_binary_hash_wj_512.pkl"
CKPT_PATH = "/tmp/best_sota_deep_binary_hash_wj_512.pt"
max_pos = 30
margin = 0.2


device=cuda:0


In [2]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
qt_norm = l1_simplex(qt.copy())


dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)


In [3]:
def wj_torch(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

class PairDataset(Dataset):
    def __init__(self, qt, gt, query_start, max_pos=30):
        self.vecs = torch.tensor(qt, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}")
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        rid = random.randrange(0, query_start)
        return self.vecs[qid], self.vecs[pid], self.vecs[rid]

def embed_all(model, qt, batch_size=512):
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(qt), batch_size):
            x = torch.tensor(qt[start:start + batch_size], dtype=torch.float32, device=device)
            out.append(model.encode(x).detach().cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    print(f"embs={embs.shape} | mem={corpus_embs.nbytes/1024**2:.1f} MB")
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim, "vec_mb": corpus_embs.nbytes/1024**2}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    if run_rerank:
        for ck in candidate_ks:
            cand, cand_info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
            t0 = time.time()
            rr = rerank_raw_wj_numpy(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
            qps_total = len(query_qt) / max(time.time() - t0 + len(query_qt)/max(cand_info['qps'], 1e-9), 1e-9)
            rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
            key = f"{method_name}_rerank_{ck}"
            for k, v in rr_metrics.items():
                if isinstance(k, int): print(f"{key} R@{k:<4} = {v:.4f}")
            save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})


In [4]:
class BinaryHashMLP(nn.Module):
    def __init__(self, in_dim, bits=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, bits),
        )
    def encode_prob(self, x):
        return torch.sigmoid(self.net(x))
    def encode(self, x):
        p = self.encode_prob(x)
        b = (p > 0.5).float()
        return b
    def forward(self, x):
        p = self.encode_prob(x)
        b = (p > 0.5).float()
        return b.detach() - p.detach() + p, p

def bit_sim(a, b):
    return 1.0 - torch.abs(a - b).mean(dim=1)

model = BinaryHashMLP(qt.shape[1], out_dim).to(device)
dataset = PairDataset(qt_norm, gt, query_start, max_pos=max_pos)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
best = float('inf')
for epoch in range(1, epochs + 1):
    model.train(); total = 0; steps = 0
    for a, p, r in loader:
        a = a.to(device, non_blocking=True); p = p.to(device, non_blocking=True); r = r.to(device, non_blocking=True)
        b, prob = model(torch.cat([a, p, r], dim=0))
        ba, bp, br = b.chunk(3, dim=0)
        sim_ap = bit_sim(ba, bp)
        sim_ar = bit_sim(ba, br)
        balance = (prob.mean(dim=0) - 0.5).pow(2).mean()
        quant = (prob - 0.5).abs().mean()
        loss = F.relu(sim_ar - sim_ap + margin).mean() + 0.1 * balance - 0.01 * quant
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        total += float(loss.detach()); steps += 1
    avg = total / max(steps, 1)
    if avg < best:
        best = avg; torch.save(model.state_dict(), CKPT_PATH)
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} loss={avg:.4f}")
print(f"best={best:.4f} saved {CKPT_PATH}")


pairs=46,722
epoch 01/30 loss=0.0083
epoch 05/30 loss=0.0034
epoch 10/30 loss=0.0022
epoch 15/30 loss=0.0017
epoch 20/30 loss=0.0015
epoch 25/30 loss=0.0010
epoch 30/30 loss=0.0009
best=0.0007 saved /tmp/best_sota_deep_binary_hash_wj_512.pt


In [5]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
model.eval()
chunks = []
with torch.no_grad():
    for start in range(0, len(qt_norm), 512):
        x = torch.tensor(qt_norm[start:start + 512], dtype=torch.float32, device=device)
        chunks.append(model.encode(x).cpu().numpy().astype(np.uint8))
bits = np.vstack(chunks)
corpus_bits = bits[:query_start]
query_bits = bits[query_start:]
print(f"bits={bits.shape} | unpacked corpus mem={corpus_bits.nbytes/1024**2:.1f} MB")

def hamming_topk(query_bits, corpus_bits, k=500, batch_size=32):
    nbrs = []
    t0 = time.time()
    for start in range(0, len(query_bits), batch_size):
        qb = query_bits[start:start + batch_size]
        sim = (qb[:, None, :] == corpus_bits[None, :, :]).mean(axis=2)
        kk = min(k, len(corpus_bits))
        part = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        rows = np.arange(len(qb))[:, None]
        order = np.argsort(-sim[rows, part], axis=1)
        top = part[rows, order]
        nbrs.extend([row.tolist() for row in top])
    qps = len(query_bits) / max(time.time() - t0, 1e-9)
    return nbrs, qps

max_k = 500
nbrs, qps = hamming_topk(query_bits, corpus_bits, k=max_k)
metrics = {**eval_recall(gt, nbrs, query_start, max_k), "qps": qps, "bits": out_dim, "vec_mb_unpacked": corpus_bits.nbytes/1024**2}
for k, v in metrics.items():
    if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
print(f"QPS={qps:.1f}")
save_result(OUT_PATH, dataset_name, METHOD_NAME, metrics, meta={"notebook": NOTEBOOK_NAME})
cleanup()


bits=(10000, 512) | unpacked corpus mem=3.9 MB
R@10   = 0.4322
R@50   = 0.6519
R@100  = 0.7421
R@500  = 0.9389
QPS=328.0
saved deep_binary_hash_wj_512 -> /tmp/results_sota_deep_binary_hash_wj_512.pkl
